## Learn delta advanced merge into and upsert_pipeline

### Merge into
在旧时代的大数据架构中，要实现“增量更新（Upsert：有则更新，无则插入）”是一场极其低效的毁灭性灾难。

因为传统的 HDFS Parquet 文件是不可变的。为了把上游增量进来的几百条卖家新数据“合”进几亿条的历史大表里，你必须把全表几亿行的数据全部读进内存，和这几百条数据做全量 LEFT JOIN，然后把几亿行数据【重新洗牌（Shuffle）并全表覆盖重写落盘】。这就好比为了换书架上的一本书，你非要把整栋图书馆给拆了重建！

Delta Lake 的 MERGE INTO 机制彻底终结了这种算子过载。它凭借底层的【_delta_log 事务账本】和【文件级别控制】技术，让引擎具备了手术刀级别的物理精准度。
它在底层只会去精准改动那几个被触及到的 Parquet 文件，吐出包含新状态的块，并在账本里做逻辑替换。其余没有被波及的历史 Parquet 文件，在物理硬盘上雷打不动，连一纳米的网线流量都不需要消耗！


### Experiment A: 构建实验时空（初始化历史大表 vs 每日增量大军）
为了严密测试“新增”和“更新”的物理闭环，我们先准备两个 DataFrame：一个是基础历史表，另一个是今日增量（包含全新卖家，也包含老卖家改名、改价格）

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *

# 初始化底座老数据，老卖家状态
columns = ["seller_id","seller_name","total_sales","last_update_date"]

history_data = [
    (101, "yuto",888888.0,"2026-06-08"),
    (102, "torna",26000.0,"2026-06-10"),
    (103, "Davf",300000.0,"2026-06-14")
]

# 由于spark机制，它会将last_update_date自动识别成string，可以用其他方法，例如用‘显示声明表结构方法’：schema = StructType([...])


df_history = spark.createDataFrame(history_data,columns)
display(df_history)
df_history.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_seller_dimension")

In [0]:
# 手动创建增量

# 101 改名且业绩上涨（触发 UPDATE）；102 业绩没有变化；104 是今天新诞生的全新卖家（触发INSERT）
incremental_data = [
    (101, "Yuto", 888888.8, "2026-06-14"),  # 现有卖家更新
    (104, "Nordic AI", 12000.0, "2026-06-14")   # 全新卖家入库
]
df_incremental = spark.createDataFrame(incremental_data,columns)
display(df_incremental)
df_incremental.createOrReplaceTempView("src_incremental_view")

### 然后我们需要用sql语句写upsert固化模版：

1. 选SQL的核心理由
MERGE 语句天生就是 SQL 语法
Upsert有则更新、无则插入是数仓标准DML作，SQL写法最直观、简洁、可读性强。如果用纯PySpark代码实现同一份UPSERT，代码会冗长很多。

2. 数仓开发习惯
分层（bronze/silver/gold）、维度表 / 事实表、视图、合并更新，是数据仓库常规任务，数据分析师 / 数仓工程师主流就用 SQL。

3. 执行 & 维护简单
不用关心 DataFrame 转换、类型映射，只关注业务字段、关联条件、更新规则，新人接手也能快速看懂。
总结：做表级别的增量合并、查询、统计 → 优先 SQL；做复杂清洗、特征计算、循环逻辑 → 优先 PySpark。

其次，为什么被称作固化模版？
简单来说，就是可以通过简单改改，日后就可以被同类表快速复用

在这里 merge into ... using ... on ... when matched ... then update set ... when not matched then insert.... 就是固化的，工程师再根据业务需求改里面的内容

In [0]:
%sql
-- 固化升级upsert模版
MERGE INTO gold_seller_dimension AS tgt
USING src_incremental_view AS src
ON tgt.seller_id = src.seller_id

WHEN MATCHED AND(tgt.seller_name != src.seller_name OR tgt.total_sales != src.total_sales OR tgt.last_update_date != src.last_update_date) THEN UPDATE SET
    tgt.seller_name = src.seller_name,
    tgt.total_sales = src.total_sales,
    tgt.last_update_date = src.last_update_date

WHEN NOT MATCHED THEN 
    INSERT(seller_id, seller_name, total_sales, last_update_date)
    VALUES(src.seller_id, src.seller_name, src.total_sales, src.last_update_date) 



In [0]:
print("=== 🏥 正在启动 Upsert 全时空物理对账流水线 ===\n")

# 1. 抓取合并后的最终全貌
df_result = spark.table("gold_seller_dimension").sort("seller_id")
df_result.show()

# 🔍 校验点 A：验证全新卖家 104 是否成功物理入库（新增逻辑测试）
assert df_result.filter("seller_id = 104").count() == 1, "🚨 警报！全新客流 104 丢失，INSERT 逻辑失败！"
print("✅ [测试通过] 104 号新客成功打入磁道，且数据结构完美！")

# 🔍 校验点 B：验证老客 101 是否改名且业绩更新（更新逻辑测试）
seller_101 = df_result.filter("seller_id = 101").collect()[0]
assert seller_101["seller_name"] == "Yuto" and seller_101["total_sales"] == 888888.8, "🚨 警报！101 状态未更新，UPDATE 逻辑未生效！"
print("✅ [测试通过] 101 号老客状态被手术刀级精准更新，无副作用！")

# 🔍 校验点 C：铁证审计！验证未被波及的历史数据（102 和 103）是否真的完好无损、保持原样
seller_102 = df_result.filter("seller_id = 102").collect()[0]
seller_103 = df_result.filter("seller_id = 103").collect()[0]
assert seller_102["last_update_date"] == "2026-06-10" and seller_103["last_update_date"] == "2026-06-14", "🚨 警报！未变更的历史数据遭到物理篡改或污染！"
print("✅ [测试通过] 102、103 号历史静态文件稳如磐石，真实验证历史数据未受一纳米污染！")

print("\n🎉 [审计结论]高级 MERGE INTO 增量管道模版全线固化成功，数据时空完美闭环！")